# AI Automation Workflows - Experimentation Notebook

This notebook provides an interactive environment for experimenting with the AI automation workflows. You can test different agents, workflows, and configurations here.

## Setup

First, let's import the necessary modules and set up the environment.

In [ ]:
# Install dependencies if needed
# !pip install -r ../requirements.txt

# Add src to path
import sys
import os
from pathlib import Path
sys.path.insert(0, str(Path().absolute().parent / "src"))

# Import modules
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Import AI automation modules
from src.utils.llm_client import LLMClientFactory, LLMConfig
from src.agents.email_agent import EmailAgent, Email
from src.agents.report_agent import ReportAgent, ReportData
from src.agents.summarizer import SummarizerAgent, SummaryType, Document
from src.workflows.automate_reporting import AutomatedReportingWorkflow, ReportSchedule
from src.workflows.customer_support_flow import CustomerSupportWorkflow

# Set up plotting
%matplotlib inline
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Setup complete!")

## LLM Client Configuration

Configure the LLM client for experimentation. You can use OpenAI, Anthropic, or a local model.

In [ ]:
# Check available environment variables
print("Environment Variables:")
print(f"OPENAI_API_KEY: {'✓' if os.getenv('OPENAI_API_KEY') else '✗'}")
print(f"ANTHROPIC_API_KEY: {'✓' if os.getenv('ANTHROPIC_API_KEY') else '✗'}")

# Initialize LLM client
try:
    llm_client = LLMClientFactory.create_from_env()
    print(f"✓ LLM client initialized successfully")
    print(f"Provider: {llm_client.config.provider}")
    print(f"Model: {llm_client.config.model_name}")
except Exception as e:
    print(f"✗ Failed to initialize LLM client: {e}")
    print("Please set up your API keys in environment variables")
    # For testing without API keys, you can use a mock client
    class MockLLMClient:
        def generate_response(self, prompt, **kwargs):
            return "This is a mock response for testing purposes."
        
        def generate_structured_response(self, prompt, schema):
            return {"mock": "response"}
    
    llm_client = MockLLMClient()
    print("Using mock LLM client for demonstration")

## Email Agent Experiments

Test the email agent with various email samples.

In [ ]:
# Initialize Email Agent
email_agent = EmailAgent(llm_client)

# Sample emails for testing
sample_emails = [
    Email(
        sender="urgent@customer.com",
        subject="URGENT: System Down - Critical Issue",
        body="Our entire system is down and we're losing money every minute. This needs immediate attention!",
        timestamp=datetime.now().isoformat()
    ),
    Email(
        sender="happy@customer.com",
        subject="Thank you for excellent service!",
        body="I just wanted to say how happy I am with your product. It's amazing and has transformed our business.",
        timestamp=datetime.now().isoformat()
    ),
    Email(
        sender="info@newsletter.com",
        subject="Weekly Newsletter",
        body="Here's our weekly newsletter with updates and news.",
        timestamp=datetime.now().isoformat()
    )
]

# Process emails
print("=== Email Processing Results ===")
for i, email in enumerate(sample_emails, 1):
    print(f"\n--- Email {i} ---")
    print(f"From: {email.sender}")
    print(f"Subject: {email.subject}")
    
    # Classify email
    classification = email_agent.classify_email(email)
    print(f"Classification: {classification}")
    
    # Detect sentiment
    sentiment = email_agent.detect_sentiment(email)
    print(f"Sentiment: {sentiment}")
    
    # Extract key info
    key_info = email_agent.extract_key_info(email)
    print(f"Key Info: {key_info}")
    
    # Generate summary
    summary = email_agent.summarize_email(email)
    print(f"Summary: {summary}")

## Report Agent Experiments

Test the report agent with sample data.

In [ ]:
# Initialize Report Agent
report_agent = ReportAgent(llm_client)

# Generate sample data
dates = pd.date_range(start='2024-01-01', end='2024-01-31', freq='D')
np.random.seed(42)  # For reproducible results

sample_data = pd.DataFrame({
    'date': dates,
    'revenue': np.random.normal(1000, 200, len(dates)) + np.arange(len(dates)) * 10,
    'customers': np.random.normal(50, 10, len(dates)) + np.arange(len(dates)) * 0.5,
    'orders': np.random.normal(100, 20, len(dates)) + np.arange(len(dates)) * 1,
    'conversion_rate': np.random.beta(2, 18, len(dates)) + 0.05
})

print("Sample Data Shape:", sample_data.shape)
print("\nSample Data Head:")
print(sample_data.head())

# Visualize the data
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Sample Business Metrics', fontsize=16)

sample_data.plot(x='date', y='revenue', ax=axes[0,0], title='Revenue Over Time')
sample_data.plot(x='date', y='customers', ax=axes[0,1], title='Customers Over Time')
sample_data.plot(x='date', y='orders', ax=axes[1,0], title='Orders Over Time')
sample_data.plot(x='date', y='conversion_rate', ax=axes[1,1], title='Conversion Rate Over Time')

plt.tight_layout()
plt.show()

In [ ]:
# Analyze trends in the data
print("=== Trend Analysis ===")
trends = report_agent.analyze_data_trends(sample_data)
for key, value in trends.items():
    print(f"{key}: {value}")

# Identify key insights
print("\n=== Key Insights ===")
insights = report_agent.identify_key_insights(sample_data)
for i, insight in enumerate(insights, 1):
    print(f"{i}. {insight}")

# Generate recommendations
print("\n=== Recommendations ===")
business_objectives = ["Increase Revenue", "Improve Customer Satisfaction", "Reduce Costs"]
recommendations = report_agent.create_recommendations(sample_data, business_objectives)
for i, rec in enumerate(recommendations, 1):
    print(f"{i}. {rec}")

# Detect anomalies
print("\n=== Anomaly Detection ===")
anomalies = report_agent.detect_anomalies(sample_data, threshold=2.0)
if anomalies:
    print(f"Found {len(anomalies)} anomalies:")
    for anomaly in anomalies[:5]:  # Show first 5
        print(f"- {anomaly['column']}: {anomaly['value']} (Z-score: {anomaly['z_score']:.2f})")
else:
    print("No significant anomalies detected.")

In [ ]:
# Generate a comprehensive report
report_data = ReportData(
    title="Monthly Performance Analysis",
    data_source="sample_business_data.csv",
    metrics={
        "total_revenue": float(sample_data['revenue'].sum()),
        "avg_daily_revenue": float(sample_data['revenue'].mean()),
        "total_customers": int(sample_data['customers'].sum()),
        "total_orders": int(sample_data['orders'].sum()),
        "avg_conversion_rate": float(sample_data['conversion_rate'].mean()),
        "revenue_growth": float((sample_data['revenue'].iloc[-1] - sample_data['revenue'].iloc[0]) / sample_data['revenue'].iloc[0] * 100)
    },
    time_period="2024-01",
    generated_at=datetime.now()
)

report = report_agent.generate_report(report_data)

print("=== Generated Report ===")
print(f"Title: {report['title']}")
print(f"\nExecutive Summary:")
print(report['executive_summary'])
print(f"\nKey Insights ({len(report['key_insights'])}):")
for i, insight in enumerate(report['key_insights'], 1):
    print(f"{i}. {insight}")
print(f"\nRecommendations ({len(report['recommendations'])}):")
for i, rec in enumerate(report['recommendations'], 1):
    print(f"{i}. {rec}")

## Summarizer Agent Experiments

Test the summarizer agent with different types of content.

In [ ]:
# Initialize Summarizer Agent
summarizer = SummarizerAgent(llm_client)

# Sample long text for summarization
long_article = """
Artificial Intelligence (AI) has become one of the most transformative technologies of the 21st century. 
From healthcare to finance, from education to entertainment, AI is reshaping industries and creating 
new possibilities that were once thought impossible. The rapid advancement of machine learning algorithms, 
particularly deep learning, has enabled computers to perform tasks that previously required human intelligence.

In healthcare, AI-powered diagnostic tools are helping doctors detect diseases like cancer earlier and 
more accurately than ever before. Machine learning models can analyze medical images, patient records, 
and genetic data to identify patterns that might be invisible to the human eye. This not only improves 
patient outcomes but also reduces healthcare costs by enabling earlier interventions.

The financial industry has embraced AI for fraud detection, algorithmic trading, risk assessment, and 
customer service. AI systems can analyze millions of transactions in real-time to identify suspicious 
patterns and prevent fraudulent activities. Robo-advisors use AI to provide personalized investment advice 
to individual investors, making sophisticated financial planning accessible to everyone.

In the automotive industry, AI is the driving force behind autonomous vehicles. Companies like Tesla, 
Waymo, and Cruise are developing self-driving cars that use AI to navigate complex traffic situations, 
make split-second decisions, and ensure passenger safety. These vehicles use a combination of computer 
vision, sensor fusion, and deep learning to understand their environment and respond appropriately.

However, the rapid advancement of AI also raises important ethical and societal questions. Issues of 
bias in AI systems, data privacy, job displacement, and the concentration of power in a few tech 
companies need to be addressed. Researchers and policymakers are working to develop frameworks for 
responsible AI development and deployment.

Looking to the future, AI continues to evolve at an exponential pace. Emerging technologies like 
quantum computing could dramatically accelerate AI capabilities, while advances in natural language 
processing are making AI systems more conversational and intuitive. As we stand on the brink of 
what many call the AI revolution, it's clear that artificial intelligence will continue to transform 
our world in ways we're only beginning to understand.
"""

print("=== Text Summarization Experiments ===")
print(f"Original text length: {len(long_article)} characters")

# Generate different types of summaries
summary_types = [SummaryType.EXECUTIVE, SummaryType.DETAILED, SummaryType.BULLET_POINTS, SummaryType.TLDR]

for summary_type in summary_types:
    print(f"\n--- {summary_type.value.upper()} SUMMARY ---")
    summary = summarizer.summarize_text(long_article, summary_type)
    print(summary)
    print(f"Length: {len(summary)} characters")

In [ ]:
# Test document summarization
document = Document(
    title="AI in Healthcare: Transforming Patient Care",
    content=long_article,
    content_type="article",
    length=len(long_article),
    source="Tech Journal"
)

doc_summary = summarizer.summarize_document(document, SummaryType.EXECUTIVE)

print("=== Document Summary ===")
print(f"Title: {doc_summary['title']}")
print(f"\nSummary: {doc_summary['summary']}")
print(f"\nKey Points:")
for i, point in enumerate(doc_summary['key_points'], 1):
    print(f"{i}. {point}")
print(f"\nMain Themes:")
for i, theme in enumerate(doc_summary['main_themes'], 1):
    print(f"{i}. {theme}")

In [ ]:
# Test meeting transcript summarization
meeting_transcript = """
Alice: Good morning everyone. Thanks for joining today's product planning meeting.
Bob: Morning Alice. I've reviewed the Q3 metrics and we're seeing strong growth in user acquisition.
Charlie: That's great news. Our marketing campaigns seem to be working well.
Alice: Yes, but I'm concerned about user retention. Our churn rate increased by 2% last month.
Bob: I noticed that too. I think we need to improve our onboarding process.
Charlie: I agree. We should also consider adding more features to keep users engaged.
Alice: Good points. Let's make that a priority for Q4. Bob, can you work on the onboarding improvements?
Bob: Absolutely. I'll have a proposal ready by next week.
Charlie: I'll research potential new features and present options at our next meeting.
Alice: Perfect. Let's also schedule a user research session to understand why people are leaving.
Bob: Good idea. I'll coordinate with the UX team.
Charlie: Should we consider adjusting our pricing strategy?
Alice: Let's hold off on pricing changes until we address the retention issues first.
Bob: Agreed. Focus on the fundamentals first.
Alice: Great. So our action items are: Bob - onboarding proposal, Charlie - feature research, 
and Alice - user research coordination. Our next meeting is in two weeks.
Charlie: Sounds good. Thanks everyone.
Bob: Thanks Alice.
"""

participants = ["Alice", "Bob", "Charlie"]
meeting_summary = summarizer.summarize_meeting_transcript(meeting_transcript, participants)

print("=== Meeting Summary ===")
print(f"Overview: {meeting_summary['overview']}")
print(f"\nKey Decisions:")
for i, decision in enumerate(meeting_summary['key_decisions'], 1):
    print(f"{i}. {decision}")
print(f"\nAction Items:")
for i, item in enumerate(meeting_summary['action_items'], 1):
    print(f"{i}. {item}")
print(f"\nNext Steps:")
for i, step in enumerate(meeting_summary['next_steps'], 1):
    print(f"{i}. {step}")

## Workflow Experiments

Test the automated workflows with sample scenarios.

In [ ]:
# Test Automated Reporting Workflow
reporting_workflow = AutomatedReportingWorkflow(llm_client)

# Create sample data files
import os
os.makedirs("../data/sample_inputs", exist_ok=True)

# Save sample data
sample_data.to_csv("../data/sample_inputs/business_metrics.csv", index=False)

# Generate ad-hoc report
print("=== Automated Reporting Test ===")
ad_hoc_report = reporting_workflow.generate_ad_hoc_report(
    "Business Performance Report",
    ["../data/sample_inputs/business_metrics.csv"],
    "standard"
)

print(f"Report Title: {ad_hoc_report['title']}")
print(f"Executive Summary: {ad_hoc_report['executive_summary'][:200]}...")
print(f"Key Insights: {len(ad_hoc_report['key_insights'])} found")
print(f"Recommendations: {len(ad_hoc_report['recommendations'])} provided")

In [ ]:
# Test Customer Support Workflow
support_workflow = CustomerSupportWorkflow(llm_client, db_path="../data/test_support.db")

# Create sample support tickets
sample_tickets = [
    {
        "customer_email": "john.doe@example.com",
        "subject": "Cannot access my account",
        "description": "I've been trying to log in for the past hour but keep getting an error message.",
        "category": "login_issue"
    },
    {
        "customer_email": "jane.smith@company.com",
        "subject": "Billing question",
        "description": "I was charged twice for my subscription this month. Can you help fix this?",
        "category": "billing"
    },
    {
        "customer_email": "mike.wilson@startup.io",
        "subject": "Feature request",
        "description": "It would be great if you could add export functionality to the dashboard.",
        "category": "feature_request"
    }
]

print("=== Customer Support Workflow Test ===")
created_tickets = []

for ticket_data in sample_tickets:
    ticket_id = support_workflow.create_ticket(
        ticket_data["customer_email"],
        ticket_data["subject"],
        ticket_data["description"],
        ticket_data["category"]
    )
    created_tickets.append(ticket_id)
    print(f"Created ticket: {ticket_id}")

# Get dashboard metrics
metrics = support_workflow.get_dashboard_metrics()
print(f"\nSupport Dashboard Metrics:")
print(f"Total Tickets: {metrics['total_tickets']}")
print(f"Tickets by Status: {metrics['tickets_by_status']}")
print(f"Tickets by Priority: {metrics['tickets_by_priority']}")

# Generate daily report
daily_report = support_workflow.generate_daily_report()
print(f"\nDaily Report:")
print(f"Date: {daily_report['date']}")
print(f"New Tickets: {daily_report['new_tickets']}")
print(f"Resolved Tickets: {daily_report['resolved_tickets']}")
print(f"Resolution Rate: {daily_report['resolution_rate']:.1f}%")

## Performance Testing

Test the performance of different agents and workflows.

In [ ]:
import time

# Performance test for email processing
def test_email_performance(email_count=10):
    print(f"Testing email processing performance with {email_count} emails...")
    
    # Generate test emails
    test_emails = []
    for i in range(email_count):
        email = Email(
            sender=f"test{i}@example.com",
            subject=f"Test Email {i}",
            body=f"This is test email number {i} with some content to process.",
            timestamp=datetime.now().isoformat()
        )
        test_emails.append(email)
    
    # Measure processing time
    start_time = time.time()
    results = email_agent.process_inbox(test_emails)
    end_time = time.time()
    
    processing_time = end_time - start_time
    avg_time_per_email = processing_time / email_count
    
    print(f"Total processing time: {processing_time:.2f} seconds")
    print(f"Average time per email: {avg_time_per_email:.2f} seconds")
    print(f"Emails per second: {email_count / processing_time:.2f}")
    
    return processing_time

# Test different batch sizes
batch_sizes = [1, 5, 10, 20]
performance_results = {}

for batch_size in batch_sizes:
    processing_time = test_email_performance(batch_size)
    performance_results[batch_size] = processing_time

# Visualize performance results
plt.figure(figsize=(10, 6))
plt.plot(list(performance_results.keys()), list(performance_results.values()), 'bo-')
plt.xlabel('Number of Emails')
plt.ylabel('Processing Time (seconds)')
plt.title('Email Processing Performance')
plt.grid(True)
plt.show()

## Custom Experiments

Use this section to run your own experiments with the AI automation workflows.

In [ ]:
# Your custom experiments go here
# Feel free to modify and extend the agents and workflows

# Example: Test custom email classification
# custom_email = Email(
#     sender="your@test.com",
#     subject="Your Subject",
#     body="Your email content",
#     timestamp=datetime.now().isoformat()
# )
# 
# result = email_agent.process_inbox([custom_email])
# print(result)

print("Ready for your custom experiments!")

## Cleanup

Clean up any temporary files and databases created during experimentation.

In [ ]:
# Clean up test database
import os

test_db_path = "../data/test_support.db"
if os.path.exists(test_db_path):
    os.remove(test_db_path)
    print(f"Removed test database: {test_db_path}")

print("\nExperimentation session completed!")
print("Feel free to save this notebook and continue experimenting later.")